In [ ]:
import warnings
import os
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
from llama_cpp import Llama, GGML_TYPE_Q4_0
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter

# ==========================================
# 1. 參數設定 (Configuration)
# ==========================================
# 請確認以下檔案路徑
MODEL_PATH = r"models\Mistral-Nemo-Instruct-2407-Q4_K_M.gguf"
CSV_PATH   = r"data\raw\fed_transcripts_with_labels.csv"
OUT_CSV    = r"data\processed\FOMC_Analysis_Rigorous_TimeSeries.csv"

# CSV 中紀錄「未來6個月升降息」的欄位名稱
TARGET_COL = 'Rate_Change_6M' 

# RAG 設定
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2' 
TOP_K_RETRIEVAL = 3      # 參考過去最像的 3 場
READ_CHAR_LIMIT = 20000  # 讀取當前會議前 20,000 字元
RAG_INDEX_LIMIT = 3000   # 建立索引時只看前 3,000 字元

# 投票與模型設定
N_ROUNDS = 5             # 每一場投 5 票
LLM_TEMPERATURE = 0.2    # 低溫模式，讓數字更穩定
N_CTX = 32768            
N_GPU_LAYERS = -1        

warnings.filterwarnings("ignore")

# ==========================================
# 2. 模型載入 (Model Loading)
# ==========================================
print("🔄 正在初始化系統...")

if not os.path.exists(MODEL_PATH) or not os.path.exists(CSV_PATH):
    print(f"❌ 錯誤：找不到檔案！請檢查路徑。\n{MODEL_PATH}\n{CSV_PATH}")
    exit()

try:
    # 1. LLM (GPU 加速)
    print("   -> 載入 Mistral LLM (GPU)...")
    llm = Llama(
        model_path=MODEL_PATH, n_ctx=N_CTX, n_gpu_layers=N_GPU_LAYERS,
        flash_attn=True, verbose=False
    )
    
    # 2. RAG Embedding (強制 CPU)
    print("   -> 載入 Embedding 模型 (CPU Mode)...")
    embedder = SentenceTransformer(EMBEDDING_MODEL_NAME, device='cpu')
    
    print("✅ 模型全數載入成功！")
except Exception as e:
    print(f"❌ 模型初始化失敗：{e}")
    exit()

# ==========================================
# 3. 資料準備與索引 (Data Prep)
# ==========================================
print("📚 讀取資料並建立 RAG 索引...")
df = pd.read_csv(CSV_PATH)
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# 填補 Ground Truth 空值 (分析用)
df[TARGET_COL] = df[TARGET_COL].fillna(0.0)

# 建立向量資料庫
corpus_texts = df['Text'].astype(str).str[:RAG_INDEX_LIMIT].tolist()
corpus_embeddings = embedder.encode(corpus_texts, show_progress_bar=True)

# ==========================================
# 4. 強力解析與核心函數 (Modified Logic)
# ==========================================

def retrieve_references(current_idx, current_embedding, top_k=3):
    """RAG 檢索：找出過去最像的會議"""
    if current_idx < top_k: return []
    past_embeddings = corpus_embeddings[:current_idx]
    similarities = cosine_similarity([current_embedding], past_embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    refs = []
    for idx in top_indices:
        row = df.iloc[idx]
        refs.append({
            "date": row['Date'].strftime('%Y-%m-%d'),
            "outcome": row[TARGET_COL],
            "snippet": str(row['Text'])[:300]
        })
    return refs

def parse_exact_score_robust(text):
    """暴力型解析器"""
    clean_text = text.replace("*", "").replace("#", "").replace("`", "")
    match = re.search(r"(?:Score|Rating|Vote)\s*[:=]?\s*(\d+(?:\.\d+)?)", clean_text, re.IGNORECASE)
    if match:
        val = float(match.group(1))
        if 0 <= val <= 10: return val

    candidates = re.findall(r"\b\d+(?:\.\d+)?\b", clean_text)
    valid_scores = []
    for c in candidates:
        try:
            val = float(c)
            if 0 <= val <= 10:
                valid_scores.append(val)
        except: pass
        
    if valid_scores:
        return valid_scores[-1]
    return None

def run_llm_inference(sys_prompt, user_prompt, max_retries=3):
    """推論函數：包含自動重試機制"""
    for attempt in range(max_retries):
        try:
            out = llm.create_chat_completion(
                messages=[{"role":"system","content":sys_prompt}, {"role":"user","content":user_prompt}],
                max_tokens=400, temperature=LLM_TEMPERATURE
            )
            content = out['choices'][0]['message']['content']
            score = parse_exact_score_robust(content)
            if score is not None:
                return round(score, 2)
        except Exception as e: 
            pass
    return None 

def calculate_rigorous_sentiment(votes, radius=1.0, min_votes=3):
    """
    ★ 核心修改：DBCF + Smart Imputation ★
    不取平均，優先尋找共識。若無共識，則使用中位數填補。
    """
    if not votes:
        return np.nan, "FAILED (Empty Votes)"

    # 1. 計算密度 (Density Scan)
    densities = [sum(1 for x in votes if abs(x - v) <= radius) for v in votes]
    max_d = max(densities) if densities else 0
    
    # 2. 策略分流
    # Case A: 存在強共識 (>=3票) -> 走 DBCF 標準流程
    if max_d >= min_votes:
        candidates_indices = [i for i, d in enumerate(densities) if d == max_d]
        candidates = [votes[i] for i in candidates_indices]
        
        # 鎖定核心群
        center = np.median(candidates)
        cluster = [v for v in votes if abs(v - center) <= radius]
        
        # 擇優：眾數 > 中位數
        mode_val, mode_count = Counter(cluster).most_common(1)[0]
        
        if mode_count > 1:
            return mode_val, f"Rigorous Consensus (Mode {mode_count}/{len(votes)})"
        else:
            return np.median(cluster), f"Rigorous Consensus (Median of {len(cluster)})"
            
    # Case B: 共識不足 (Chaos) -> 走 Smart Imputation
    # 直接取全體中位數，捕捉當下最可能的中心傾向，避免資料中斷
    else:
        imputed_val = np.median(votes)
        return imputed_val, f"Smart Imputation (Median of {len(votes)})"

# ==========================================
# 5. 主分析迴圈 (Main Loop)
# ==========================================
SYSTEM_PROMPT = """You are a Fed Analyst. 

**SCORING SCALE (0 to 10):**
- 0 = Extreme Dovish (Cut Rates)
- 5 = Neutral
- 10 = Extreme Hawkish (Hike Rates)

**Task:**
Analyze the Transcript based on Historical References.
Provide a **Precise Score** (e.g., 7.25, 4.50).

**Output:**
Thinking: ...
Score: [number]"""

results = []
print(f"\n🚀 開始嚴謹分析 {len(df)} 場會議 (DBCF + Smart Imputation Mode)...")
print("=" * 70)

for i in tqdm(range(len(df))):
    row = df.iloc[i]
    date_str = row['Date'].strftime('%Y-%m-%d')
    current_text = str(row['Text'])[:READ_CHAR_LIMIT] 
    
    # --- Step 1: RAG 檢索 ---
    references = retrieve_references(i, corpus_embeddings[i], top_k=TOP_K_RETRIEVAL)
    
    ref_block = ""
    if references:
        ref_block = "--- Historical References (Context) ---\n"
        for ref in references:
            ref_block += f"[Date: {ref['date']}] -> Next 6M Change: {ref['outcome']:+.2f}%\nSnippet: {ref['snippet']}...\n\n"
    else:
        ref_block = "(No history available)\n"

    user_prompt = f"Date: {date_str}\n\n{ref_block}\n--- Transcript ---\n{current_text}\n\nAnalyze and give a precise Score."

    # --- Step 2: 投票 (5 回合) ---
    votes = []
    for _ in range(N_ROUNDS):
        score = run_llm_inference(SYSTEM_PROMPT, user_prompt)
        if score is not None: votes.append(score)

    # --- Step 3: 嚴謹結算 (New Logic) ---
    # 直接在此處調用新函數，無需後處理
    final_score, note = calculate_rigorous_sentiment(votes)
    
    # --- Step 4: 紀錄 ---
    results.append({
        "Date": date_str,
        "Score": final_score,         # 這是經過嚴謹處理的最終分數
        "Vote_Details": str(votes),
        "Vote_Logic": note,
        "Actual_Change_6M": row[TARGET_COL],
        "Reference_Dates": str([r['date'] for r in references])
    })
    
    if i % 5 == 0:
        # 顯示當前進度與判定邏輯 (方便監控是否有 Imputation 發生)
        tqdm.write(f" -> {date_str} : {final_score} [{note}]")

# ==========================================
# 6. 存檔與最終檢查
# ==========================================
final_df = pd.DataFrame(results)

# 最後一道防線：如果因為 LLM 完全崩潰導致 votes 為空，這裡做 Forward Fill
if final_df['Score'].isna().sum() > 0:
    print(f"⚠️ 發現 {final_df['Score'].isna().sum()} 筆全空資料，執行 Forward Fill 修補...")
    final_df['Score'] = final_df['Score'].fillna(method='ffill')

final_df.to_csv(OUT_CSV, index=False)
print("\n" + "="*70)
print(f"✅ 分析完成！嚴謹時間序列已生成：\n{OUT_CSV}")
print("此檔案中的 'Score' 欄位即為可直接投入模型的 FOMC_Sentiment 指標。")

🔄 正在初始化系統...
   -> 載入 Mistral LLM (GPU)...


llama_new_context_with_model: n_ctx_per_seq (32768) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


   -> 載入 Embedding 模型 (CPU Mode)...
✅ 模型全數載入成功！
📚 讀取資料並建立 RAG 索引...


Batches:   0%|          | 0/3 [00:00<?, ?it/s]


🚀 開始嚴謹分析 77 場會議 (DBCF + Smart Imputation Mode)...


...

 -> 2025-01-29 : 6.5 [Rigorous Consensus (Mode 4/5)]


 99%|█████████████████████████████████████████████████████████████████████████████▉ | 76/77 [2:59:03<02:12, 132.14s/it]

 -> 2025-09-17 : 6.5 [Rigorous Consensus (Mode 3/5)]


100%|███████████████████████████████████████████████████████████████████████████████| 77/77 [3:01:06<00:00, 141.12s/it]


✅ 分析完成！嚴謹時間序列已生成：
C:\Users\user\Desktop\llmcourse\FOMC_Analysis_Final_Robust.csv
此檔案中的 'Score' 欄位即為可直接投入模型的 FOMC_Sentiment 指標。
